# Modelo RF con metodo por Transecto y metodo General

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código 7 (adaptado a transectos): Modelo Random Forest.
Entrada: datasets 2D generados en código 6 (ml_2d) para transectos y global.
Salida: modelos entrenados, métricas (globales y por estación dentro de cada transecto),
        gráficas y archivos de resultados.
"""

import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

BASE_DIR = os.path.expanduser("/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/")
DATA_DIR = os.path.join(BASE_DIR, "windows_partitioned")
ENCODED_DIR = os.path.join(BASE_DIR, "encoded")  # para cargar CSV originales (necesario para estaciones)
OUTPUT_DIR = os.path.join(BASE_DIR, "models", "random_forest")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Parámetros
WINDOW_IN = 72
WINDOW_OUT = 72
RANDOM_STATE = 42
N_JOBS = -1

# Hiperparámetros para búsqueda (GridSearch)
RF_PARAM_GRID = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

# ============================================================================
# MÉTRICAS
# ============================================================================

def willmott_index(y_true, y_pred):
    numer = np.sum((y_true - y_pred) ** 2)
    denom = np.sum((np.abs(y_pred - y_true.mean()) + np.abs(y_true - y_true.mean())) ** 2)
    return 1 - numer / denom if denom != 0 else np.nan

def mape(y_true, y_pred):
    mask = y_true != 0
    if not mask.any():
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def compute_metrics(y_true, y_pred):
    return {
        'r2': r2_score(y_true, y_pred),
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mape': mape(y_true, y_pred),
        'willmott': willmott_index(y_true, y_pred)
    }

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def plot_predictions(y_true, y_pred, horizons, save_path, title):
    n_plots = len(horizons)
    fig, axes = plt.subplots(1, n_plots, figsize=(5*n_plots, 4))
    if n_plots == 1:
        axes = [axes]
    for ax, h in zip(axes, horizons):
        ax.scatter(y_true[:, h], y_pred[:, h], alpha=0.3, s=10)
        ax.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=1)
        ax.set_xlabel('Real O3 (µg/m³)')
        ax.set_ylabel('Predicho O3 (µg/m³)')
        ax.set_title(f'Horizonte {h+1}h')
        ax.grid(True, alpha=0.3)
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()

def get_station_from_features(X_sample, feature_names, station_prefix='Estacion_'):
    """
    Para una muestra X_sample (vector aplanado de longitud window_in * n_features),
    determina la estación a partir de las columnas dummy (one-hot) en el primer bloque temporal.
    feature_names: lista de nombres de las columnas (orden original, antes de aplanar).
    Retorna el nombre de la estación (string) o None si no se puede determinar.
    """
    n_features = len(feature_names)
    # Primer bloque temporal (primeras n_features posiciones)
    first_block = X_sample[:n_features]
    # Identificar índices de columnas que son dummies de estación
    station_indices = [i for i, name in enumerate(feature_names) if name.startswith(station_prefix)]
    if not station_indices:
        return None
    # Buscar cuál tiene valor 1 (aproximadamente)
    for idx in station_indices:
        if abs(first_block[idx] - 1.0) < 0.1:  # tolerancia por posibles errores numéricos
            # Extraer nombre de estación quitando el prefijo
            station_name = feature_names[idx][len(station_prefix):]
            return station_name
    return None

def compute_per_station_metrics(y_true, y_pred, X_test, feature_names, output_dir, entity_name):
    """
    Calcula métricas desglosadas por estación para un conjunto de test.
    X_test: array 2D (n_samples, window_in * n_features)
    feature_names: lista de nombres de características (mismo orden que las columnas)
    Guarda un CSV con métricas por estación y devuelve dict.
    """
    n_samples = len(y_true)
    station_preds = {}  # dict: station -> list of (y_true, y_pred) arrays (por muestra)
    
    for i in range(n_samples):
        station = get_station_from_features(X_test[i], feature_names)
        if station is None:
            continue
        if station not in station_preds:
            station_preds[station] = {'true': [], 'pred': []}
        station_preds[station]['true'].append(y_true[i])
        station_preds[station]['pred'].append(y_pred[i])
    
    # Calcular métricas por estación
    station_metrics = {}
    for station, data in station_preds.items():
        true_stack = np.vstack(data['true'])  # (n_samples_station, 72)
        pred_stack = np.vstack(data['pred'])
        # Métricas globales para esta estación (sobre todas las horas)
        metrics = compute_metrics(true_stack.ravel(), pred_stack.ravel())
        station_metrics[station] = metrics
    
    # Guardar CSV
    if station_metrics:
        df = pd.DataFrame(station_metrics).T
        df.index.name = 'station'
        df.to_csv(os.path.join(output_dir, f"{entity_name}_per_station_metrics.csv"))
        print(f"    Métricas por estación guardadas en {entity_name}_per_station_metrics.csv")
    return station_metrics

def train_and_evaluate_rf(X_train, y_train, X_val, X_test, y_val, y_test,
                          entity_name, output_subdir, feature_names=None, original_csv_path=None):
    """
    Entrena Random Forest con búsqueda de hiperparámetros.
    Si feature_names y original_csv_path se proporcionan (para transectos),
    se calculan métricas desglosadas por estación.
    """
    print(f"\n--- Entrenando Random Forest para {entity_name} ---")

    # 1. Búsqueda de hiperparámetros
    tscv = TimeSeriesSplit(n_splits=3)
    rf = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1)
    grid_search = GridSearchCV(
        estimator=rf,
        param_grid=RF_PARAM_GRID,
        cv=tscv,
        scoring='neg_mean_squared_error',
        n_jobs=N_JOBS,
        verbose=2
    )
    print("  Buscando mejores hiperparámetros...")
    grid_search.fit(X_train, y_train)

    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_
    print(f"  Mejores parámetros: {best_params}")

    # 2. Evaluación en validación
    val_pred = best_model.predict(X_val)
    val_metrics = compute_metrics(y_val.ravel(), val_pred.ravel())
    print(f"  Métricas en validación: R2={val_metrics['r2']:.3f}, MAE={val_metrics['mae']:.2f}")

    # 3. Evaluación en test
    test_pred = best_model.predict(X_test)
    test_metrics = compute_metrics(y_test.ravel(), test_pred.ravel())
    print(f"  Métricas en test: R2={test_metrics['r2']:.3f}, MAE={test_metrics['mae']:.2f}, "
          f"RMSE={test_metrics['rmse']:.2f}, MAPE={test_metrics['mape']:.2f}%, Willmott={test_metrics['willmott']:.3f}")

    # 4. Métricas por estación (si es aplicable)
    station_metrics = None
    if feature_names is not None and original_csv_path is not None:
        # Necesitamos los nombres de las características para identificar estaciones
        # feature_names debe ser la lista de nombres en el orden de las columnas (antes de aplanar)
        station_metrics = compute_per_station_metrics(y_test, test_pred, X_test, feature_names,
                                                       output_subdir, entity_name)

    # 5. Guardar modelo y resultados
    model_path = os.path.join(output_subdir, "model.pkl")
    with open(model_path, 'wb') as f:
        pickle.dump(best_model, f)

    results = {
        'best_params': best_params,
        'validation_metrics': val_metrics,
        'test_metrics': test_metrics,
        'station_metrics': station_metrics,
        'n_train': len(X_train),
        'n_val': len(X_val),
        'n_test': len(X_test)
    }
    with open(os.path.join(output_subdir, "results.json"), 'w') as f:
        json.dump(results, f, indent=2)

    # 6. Gráficas
    horizons = [23, 47, 71]
    plot_predictions(y_test, test_pred, horizons,
                     save_path=os.path.join(output_subdir, "test_scatter.png"),
                     title=f"Random Forest - {entity_name} - Test")

    np.save(os.path.join(output_subdir, "test_pred.npy"), test_pred)
    np.save(os.path.join(output_subdir, "test_true.npy"), y_test)

    return best_model, test_metrics

# ============================================================================
# PROCESAMIENTO POR TRANSECTO
# ============================================================================

def process_by_transect():
    print("\n" + "="*50)
    print("PROCESANDO MODELOS POR TRANSECTO")
    print("="*50)

    ml_2d_dir = os.path.join(DATA_DIR, "by_transect", "ml")
    if not os.path.exists(ml_2d_dir):
        print(f"No existe la carpeta: {ml_2d_dir}")
        return

    entities = [d for d in os.listdir(ml_2d_dir) if os.path.isdir(os.path.join(ml_2d_dir, d))]
    if not entities:
        print("No se encontraron transectos.")
        return

    for entity in entities:
        entity_path = os.path.join(ml_2d_dir, entity)
        ml_2d_path = os.path.join(entity_path, "ml_2d")
        if not os.path.exists(ml_2d_path):
            print(f"  {entity}: no tiene ml_2d, se omite.")
            continue

        # Cargar datos
        X_train = np.load(os.path.join(ml_2d_path, "train_X.npy"))
        y_train = np.load(os.path.join(ml_2d_path, "train_y.npy"))
        X_val = np.load(os.path.join(ml_2d_path, "val_X.npy"))
        y_val = np.load(os.path.join(ml_2d_path, "val_y.npy"))
        X_test = np.load(os.path.join(ml_2d_path, "test_X.npy"))
        y_test = np.load(os.path.join(ml_2d_path, "test_y.npy"))

        # Para obtener los nombres de las características (necesarios para desglose por estación),
        # cargamos el CSV original de este transecto (de la carpeta encoded/ml/by_transect)
        original_csv = os.path.join(ENCODED_DIR, "ml", "by_transect", f"{entity}.csv")
        feature_names = None
        if os.path.exists(original_csv):
            # Leer solo las columnas (sin cargar todos los datos)
            df_cols = pd.read_csv(original_csv, nrows=0, index_col=0)
            feature_names = df_cols.columns.tolist()
        else:
            print(f"  Advertencia: no se encontró el CSV original para {entity}. No se podrá desglosar por estación.")

        # Crear subcarpeta de salida
        out_subdir = os.path.join(OUTPUT_DIR, "by_transect", entity)
        os.makedirs(out_subdir, exist_ok=True)

        try:
            train_and_evaluate_rf(X_train, y_train, X_val, X_test, y_val, y_test,
                                  entity, out_subdir,
                                  feature_names=feature_names,
                                  original_csv_path=original_csv if os.path.exists(original_csv) else None)
        except Exception as e:
            print(f"Error procesando {entity}: {e}")
            continue

# ============================================================================
# PROCESAMIENTO GLOBAL (por estación)
# ============================================================================

def process_global():
    print("\n" + "="*50)
    print("PROCESANDO MODELOS GLOBALES (POR ESTACIÓN)")
    print("="*50)

    ml_2d_dir = os.path.join(DATA_DIR, "global", "ml")
    if not os.path.exists(ml_2d_dir):
        print(f"No existe la carpeta: {ml_2d_dir}")
        return

    entities = [d for d in os.listdir(ml_2d_dir) if os.path.isdir(os.path.join(ml_2d_dir, d))]
    if not entities:
        print("No se encontraron estaciones.")
        return

    for entity in entities:
        entity_path = os.path.join(ml_2d_dir, entity)
        ml_2d_path = os.path.join(entity_path, "ml_2d")
        if not os.path.exists(ml_2d_path):
            print(f"  {entity}: no tiene ml_2d, se omite.")
            continue

        X_train = np.load(os.path.join(ml_2d_path, "train_X.npy"))
        y_train = np.load(os.path.join(ml_2d_path, "train_y.npy"))
        X_val = np.load(os.path.join(ml_2d_path, "val_X.npy"))
        y_val = np.load(os.path.join(ml_2d_path, "val_y.npy"))
        X_test = np.load(os.path.join(ml_2d_path, "test_X.npy"))
        y_test = np.load(os.path.join(ml_2d_path, "test_y.npy"))

        out_subdir = os.path.join(OUTPUT_DIR, "global", entity)
        os.makedirs(out_subdir, exist_ok=True)

        try:
            train_and_evaluate_rf(X_train, y_train, X_val, X_test, y_val, y_test,
                                  entity, out_subdir,
                                  feature_names=None, original_csv_path=None)  # sin desglose por estación
        except Exception as e:
            print(f"Error procesando {entity}: {e}")
            continue

# ============================================================================
# RESUMEN FINAL
# ============================================================================

def generate_summary():
    """Crea un resumen de todas las métricas en un CSV."""
    summary = []

    # Por transecto
    transect_dir = os.path.join(OUTPUT_DIR, "by_transect")
    if os.path.exists(transect_dir):
        for entity in os.listdir(transect_dir):
            res_file = os.path.join(transect_dir, entity, "results.json")
            if os.path.exists(res_file):
                with open(res_file, 'r') as f:
                    data = json.load(f)
                summary.append({
                    'entity': entity,
                    'type': 'by_transect',
                    **data['test_metrics']
                })
                # Si hay station_metrics, podríamos añadirlas aparte, pero aquí solo global

    # Global
    global_dir = os.path.join(OUTPUT_DIR, "global")
    if os.path.exists(global_dir):
        for entity in os.listdir(global_dir):
            res_file = os.path.join(global_dir, entity, "results.json")
            if os.path.exists(res_file):
                with open(res_file, 'r') as f:
                    data = json.load(f)
                summary.append({
                    'entity': entity,
                    'type': 'global',
                    **data['test_metrics']
                })

    if summary:
        df = pd.DataFrame(summary)
        df.to_csv(os.path.join(OUTPUT_DIR, "summary_metrics.csv"), index=False)
        print("\nResumen guardado en:", os.path.join(OUTPUT_DIR, "summary_metrics.csv"))

# ============================================================================
# EJECUCIÓN PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    print("="*60)
    print("RANDOM FOREST - ENTRENAMIENTO Y EVALUACIÓN (TRANSECTOS + GLOBAL)")
    print("="*60)

    process_by_transect()
    process_global()
    generate_summary()

    print("\nProceso completado. Revise la carpeta:")
    print(f"  {OUTPUT_DIR}")

RANDOM FOREST - ENTRENAMIENTO Y EVALUACIÓN (TRANSECTOS + GLOBAL)

PROCESANDO MODELOS POR TRANSECTO

--- Entrenando Random Forest para Transecto_1 ---
  Buscando mejores hiperparámetros...
Fitting 3 folds for each of 162 candidates, totalling 486 fits
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=  56.6s
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time= 1.8min
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time= 2.0min
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time= 3.0min
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=300; total time= 2.7min
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time= 3.9min
[CV] END max_depth=10, max_fe